### Estrategias de optimización de bucles

**Minimizar cálculos internos (pre-cálculo de invariantes)**: Saca fuera del bucle todo lo que no dependa de la iteración actual, como una tasa de cambio fija. Así, el bucle realiza menos operaciones y corre más rápido.

**Evaluación en cortocircuito y terminación anticipada**: Usa operadores lógicos para no evaluar la segunda parte si la primera ya define el resultado, y rompa el bucle con break cuando se cumpla la meta. Esto evita iteraciones adicionales y ahorra tiempo.

**Desenrollado de bucles (loop unrolling)**: Realice varias iteraciones en una sola pasada para disminuir comprobaciones y saltos. Aunque aumenta un poco el tamaño del código, acelera la ejecución cuando las iteraciones son fijas o conocidas.

**Fusión de bucles (loop fusion)**: Combina bucles que recorren la misma colección de datos en uno solo, reduciendo accesos repetitivos a memoria y mejorando la eficiencia. Conviene fusionar solo tareas sencillas para no complicar el código.

**Separación de iteraciones (loop fission o distribución)**: Parta un bucle complejo en varios más simples, por ejemplo separando validación de escritura en base de datos. Esto mejora la claridad, la localidad de datos y, a menudo, facilita la paralelización.




In [1]:
# Ejemplo de optimización
# Calcular el monto total en USD a partir de la información del fichero montos_con_moneda.csv

#Primero recuperamos los datos en un data frame
import pandas as pd
df = pd.read_csv('montos_con_moneda.csv')

# Ahora vamos a crear una función que reciba una moneda y devuelva la tasa de cambio a USD a partir de la tabla de conversión de tabla_conversion_a_usd.csv
def tasa_cambio_a_usd(moneda):
    df_conversion = pd.read_csv('tabla_conversion_a_usd.csv')
    tasa = df_conversion.loc[df_conversion['Moneda'] == moneda, 'Tasa_Conversion_USD']
    #introduce un sleep
    import time
    time.sleep(0.1)  # Simula un retraso para optimización
    if not tasa.empty:
        return tasa.iloc[0]  # Retorna el primer valor encontrado
    else:
        raise ValueError(f'No se encontró la tasa de cambio para la moneda: {moneda}')

#Inicializamos variables
total_usd = 0.0
# Iteramos sobre cada fila del dataframe
for index, row in df.iterrows():
    monto = row['Monto']
    moneda = row['Moneda']
    
    # Obtenemos la tasa de cambio a USD
    tasa = tasa_cambio_a_usd(moneda)
  
    # Convertimos el monto a USD
    monto_usd = monto * tasa
    
    # Acumulamos el total en USD
    total_usd += monto_usd

# Imprimimos el resultado final
print(f'El monto total en USD es: {total_usd:.2f}')



El monto total en USD es: 28074.41


In [2]:
#Ejemplo de optimización simple. No calculamos tasa si la moneda es USD

#Primero recuperamos los datos en un data frame
import pandas as pd
df = pd.read_csv('montos_con_moneda.csv')

# Ahora vamos a crear una función que reciba una moneda y devuelva la tasa de cambio a USD a partir de la tabla de conversión de tabla_conversion_a_usd.csv
def tasa_cambio_a_usd(moneda):
    df_conversion = pd.read_csv('tabla_conversion_a_usd.csv')
    tasa = df_conversion.loc[df_conversion['Moneda'] == moneda, 'Tasa_Conversion_USD']
    #introduce un sleep
    import time
    time.sleep(0.1)  # Simula un retraso para optimización
    if not tasa.empty:
        return tasa.iloc[0]  # Retorna el primer valor encontrado
    else:
        raise ValueError(f'No se encontró la tasa de cambio para la moneda: {moneda}')

#Inicializamos variables
total_usd = 0.0
# Iteramos sobre cada fila del dataframe
for index, row in df.iterrows():
    monto = row['Monto']
    moneda = row['Moneda']
    
    # Si la moneda no es USD, obtenemos la tasa de cambio a USD
    if moneda != 'USD':
        tasa = tasa_cambio_a_usd(moneda)
    else:
        tasa = 1.0  # Si es USD, la tasa es 1
  
    # Convertimos el monto a USD
    monto_usd = monto * tasa
    
    # Acumulamos el total en USD
    total_usd += monto_usd

# Imprimimos el resultado final
print(f'El monto total en USD es: {total_usd:.2f}')

El monto total en USD es: 28074.41


In [3]:
#Ejemplo de optimización avanzado. Creamos una cache con las tasas de cambio para reutilizarlas

#Primero recuperamos los datos en un data frame
import pandas as pd
df = pd.read_csv('montos_con_moneda.csv')

# Ahora vamos a crear una función que reciba una moneda y devuelva la tasa de cambio a USD a partir de la tabla de conversión de tabla_conversion_a_usd.csv
def tasa_cambio_a_usd(moneda):
    df_conversion = pd.read_csv('tabla_conversion_a_usd.csv')
    tasa = df_conversion.loc[df_conversion['Moneda'] == moneda, 'Tasa_Conversion_USD']
    #introduce un sleep
    import time
    time.sleep(0.1)  # Simula un retraso para optimización
    if not tasa.empty:
        return tasa.iloc[0]  # Retorna el primer valor encontrado
    else:
        raise ValueError(f'No se encontró la tasa de cambio para la moneda: {moneda}')

#Inicializamos variables
total_usd = 0.0
tasa_cambio_a_usd_cache = {}  # Diccionario para almacenar las tasas de cambio ya calculadas

# Iteramos sobre cada fila del dataframe
for index, row in df.iterrows():
    monto = row['Monto']
    moneda = row['Moneda']
    
    # Si la moneda no es USD y no la tenemos en la cache, obtenemos la tasa de cambio a USD
    if moneda != 'USD':
        if moneda not in tasa_cambio_a_usd_cache:
            tasa = tasa_cambio_a_usd(moneda)
            tasa_cambio_a_usd_cache[moneda] = tasa  # Guardamos la tasa en la cache
        else:
            tasa = tasa_cambio_a_usd_cache[moneda]  # Usamos la tasa de la cache
    else:
        tasa = 1.0  # Si es USD, la tasa es 1
  
    # Convertimos el monto a USD
    monto_usd = monto * tasa
    
    # Acumulamos el total en USD
    total_usd += monto_usd

# Imprimimos el resultado final
print(f'El monto total en USD es: {total_usd:.2f}')

El monto total en USD es: 28074.41


### Técnicas de anidamiento

**Organizar niveles de anidamiento**: Cada nivel debe tener un propósito claro. Si hay más de 3 o 4 niveles, considere refactorizar. 

**Indentación consistente y significativa**: Aunque usemos pseudocódigo, usar sangría y FIN SI / FIN PARA de forma alineada permite identificar claramente qué está dentro de qué.

**Evitar código spaguetti**: 
1. No abusar de anidamientos profundos: a veces una estructura más plana simplifica la lectura. 
2. Evitar saltos incondicionales que rompan el flujo. 
3. Mantener bloques de tamaño razonable; dividir si crecen demasiado.


### Optimización de variables
Como ya vimos las variables en programación almacenan datos que cambian durante la ejecución. Su correcta definición y uso son esenciales para obtener resultados confiables y eficientes, sobre todo al manejar grandes volúmenes de información. Al igual que en una hoja de cálculo, etiquetar y tipificar bien las variables agiliza los cálculos y reduce el consumo de recursos, siendo clave para la eficacia de los algoritmos financieros.

**Selección eficiente del tipo de dato**: Elegir enteros para contadores, flotantes para decimales y booleanos para lógicos ayuda a prevenir errores y pérdidas de precisión, incluso en pseudocódigo.

Ejemplo: un tipo de dato string ocupa entre 2 y 6 veces más que un array.


**Gestión eficiente del ámbito (scope)**: Limitar el ámbito de las variables mejora la claridad, previene errores y optimiza el uso de memoria.

Se busca evitar cálculos repetitivos y costosos, almacenando en variables temporales los resultados intermedios para mejorar la eficiencia del algoritmo.

Ejemplo: ¿dónde declarar una variable, dentro o fuera del bucle?


**Ahorro de Recursos**: Al acumular resultados (como sumas o promedios) durante el recorrido de la lista, se evita procesar la misma información varias veces, reduciendo el consumo de tiempo y recursos.

**Simplificación del Código**: Al minimizar operaciones redundantes, se simplifica la lógica del programa, haciéndolo más fácil de mantener y comprender.Esta estrategia es esencial para optimizar el rendimiento del código y garantizar que los algoritmos se ejecuten de manera eficiente.

In [5]:
# Ejemplo. Almacenamos en una variable tipo string todas las compañías que hemos revisado ya.

# Dataframe con 10 compañías repetidas entre 3 y 4 veces aleatoriamente.
import pandas as pd

data = {
    'Compañía': [
        'Toyota', 'Samsung', 'Facebook', 'Google', 'Samsung',
        'Microsoft', 'Amazon', 'Intel', 'Apple', 'Google',
        'Coca-Cola', 'Intel', 'Facebook', 'Microsoft', 'Tesla',
        'Toyota', 'Amazon', 'Google', 'Apple', 'Coca-Cola',
        'Tesla', 'Toyota', 'Samsung', 'Intel', 'Apple',
        'Microsoft', 'Facebook', 'Amazon', 'Coca-Cola', 'Tesla'
    ]
}


# Función dummy que simula el análisis de una compañía
def analizar_compania(compania):
    # Simula un análisis que toma tiempo
    import time
    time.sleep(0.1)  # Simula un retraso para optimización
    return f"Análisis de {compania} completado."

# Recorremos el data frame y almacenamos las compañías ya revisadas en un string separado por comas
companias_revisadas = ""
df_companias = pd.DataFrame(data)
for index, row in df_companias.iterrows():
    compania = row['Compañía']
    
    # Si la compañía ya ha sido revisada, la saltamos
    if compania in companias_revisadas:
        # print(f"{compania} ya ha sido revisada. Saltando...")
        continue
    
    # Analizamos la compañía
    resultado = analizar_compania(compania)
    
    # Almacenamos el resultado en el string
    if companias_revisadas:
        companias_revisadas += ", "
    companias_revisadas += compania
    
    # Imprimimos el resultado del análisis
    print(resultado)

Análisis de Toyota completado.
Análisis de Samsung completado.
Análisis de Facebook completado.
Análisis de Google completado.
Análisis de Microsoft completado.
Análisis de Amazon completado.
Análisis de Intel completado.
Análisis de Apple completado.
Análisis de Coca-Cola completado.
Análisis de Tesla completado.


In [6]:
# Ejemplo. Almacenamos en una variable tipo lista todas las compañías que hemos revisado ya.

# Dataframe con 10 compañías repetidas entre 3 y 4 veces aleatoriamente.
import pandas as pd

data = {
    'Compañía': [
        'Toyota', 'Samsung', 'Facebook', 'Google', 'Samsung',
        'Microsoft', 'Amazon', 'Intel', 'Apple', 'Google',
        'Coca-Cola', 'Intel', 'Facebook', 'Microsoft', 'Tesla',
        'Toyota', 'Amazon', 'Google', 'Apple', 'Coca-Cola',
        'Tesla', 'Toyota', 'Samsung', 'Intel', 'Apple',
        'Microsoft', 'Facebook', 'Amazon', 'Coca-Cola', 'Tesla'
    ]
}


# Función dummy que simula el análisis de una compañía
def analizar_compania(compania):
    # Simula un análisis que toma tiempo
    import time
    time.sleep(0.1)  # Simula un retraso para optimización
    return f"Análisis de {compania} completado."

# Recorremos el data frame y almacenamos las compañías ya revisadas en una lista
companias_revisadas = []
df_companias = pd.DataFrame(data)
for index, row in df_companias.iterrows():
    compania = row['Compañía']
    
    # Si la compañía ya ha sido revisada, la saltamos
    if compania in companias_revisadas:
        continue
    
    # Analizamos la compañía
    resultado = analizar_compania(compania)
    
    # Almacenamos el resultado en la lista
    companias_revisadas.append(compania)
    
    # Imprimimos el resultado del análisis
    print(resultado)

Análisis de Toyota completado.
Análisis de Samsung completado.
Análisis de Facebook completado.
Análisis de Google completado.
Análisis de Microsoft completado.
Análisis de Amazon completado.
Análisis de Intel completado.
Análisis de Apple completado.
Análisis de Coca-Cola completado.
Análisis de Tesla completado.


In [25]:
# Ejemplo declaración de una variable

#Dentro de un bucle:
for i in range(5):
    variable = i * 2
    print(f"A. El valor de la variable en la iteración {i} es: {variable}")

# Fuera del bucle:
variable = 0
for i in range(5):
    variable += i * 2
    print(f"B. El valor de la variable en la iteración {i} es: {variable}")


A. El valor de la variable en la iteración 0 es: 0
A. El valor de la variable en la iteración 1 es: 2
A. El valor de la variable en la iteración 2 es: 4
A. El valor de la variable en la iteración 3 es: 6
A. El valor de la variable en la iteración 4 es: 8
B. El valor de la variable en la iteración 0 es: 0
B. El valor de la variable en la iteración 1 es: 2
B. El valor de la variable en la iteración 2 es: 6
B. El valor de la variable en la iteración 3 es: 12
B. El valor de la variable en la iteración 4 es: 20


### Nomenclatura de variable y buenas prácticas

**Claridad y descriptividad**: Los nombres deben expresar su propósito.

**Consistencia y convenciones de codificación**: Mantén un mismo estilo. Si decide usar español, no mezcle con variables en inglés. Referencia siempre a la variable con el mismo nombre. Esto facilita la colaboración y posterior implementación.

**Uso de prefijos o sufijos informativos**: Pueden usarse prefijos como “pct” para porcentajes o sufijos como “_Max”, “_Acum”, etc. Bien utilizados, mejoran la lectura del código. Evita nombres demasiado largos o complejos.

**Inicialización y validación de variables**: Asigne un valor inicial (0, "", etc.) antes de usarla. Valida datos de entrada (rango, tipo). Ej.: una tasa debería ser >= 0 y <= 1. Evitar operar con variables sin valor definido o con datos inválidos que distorsionen cálculos financieros.

**Control de errores y manejo de valores nulos**: Muchos datos financieros pueden no estar disponibles. Si una variable es NULO, decidir si se usa un valor por defecto o se descarta ese registro. También evitar dividir entre 0 o valores negativos en operaciones que no lo permitan. Incluir banderas de error para abortar cálculos cuando algo va mal.

**Consistencia y convenciones de codificación**: Mantén un mismo estilo. Si decide usar español, no mezcle con variables en inglés. Referencia siempre a la variable con el mismo nombre. Esto facilita la colaboración y posterior implementación.

**Modularización y reutilización**: Divida el pseudocódigo en módulos y use variables locales con nombres únicos para evitar confusiones, especialmente en bucles anidados. 

**Uso de constantes**: Defina constantes para valores fijos (como tasas o umbrales) en lugar de números mágicos, facilitando actualizaciones y reduciendo errores.

### Funciones:
Como vimos, las funciones son bloques independientes de código que realizan tareas específicas. 

**Uso fundamental para**:
1. Modularizar el código: dividir un problema grande en piezas manejables.
2. Abstraer la complejidad: ocultar detalles internos y exponer una interfaz clara.
3. Fomentar la reutilización: aplicar la misma función en distintos proyectos sin reescribir la lógica. 

**Principios y metodologías**: 
1. Definición de la Interfaz y el Comportamiento:  Establecer claramente qué datos recibe la función (parámetros) y qué devuelve (resultado).
2. Descomposición Funcional: Dividir procesos complejos en subfunciones para simplificar el problema global.
3. Modularidad y Acoplamiento: Diseñar funciones con baja dependencia (bajo acoplamiento) y que cumplan una única responsabilidad (alta cohesión). Esto permite actualizar o sustituir funciones sin afectar al sistema completo.

**Técnicas de abstracción**: 
1. Funciones Puras vs. Funciones Impuras:  Las funciones puras siempre retornan el mismo valor para un mismo conjunto de parámetros sin afectar el entorno, mientras que las funciones impuras pueden depender o modificar estados externos.
2. Recursividad y Composición de Funciones: La recursividad resuelve problemas dividiéndolos en subproblemas idénticos y la composición permite encadenar funciones, usando la salida de una como entrada de otra.
3. Funciones de Orden Superior:  Permiten devolver funciones, lo que potencia la abstracción y reutilización



In [26]:
# Ejemplo de función de orden superior. Cálculo de pares o impares.
def es_par(num):
    """Función que determina si un número es par."""
    return num % 2 == 0

def es_impar(num):
    """Función que determina si un número es impar."""
    return num % 2 != 0

def filtrar_numeros(lista, criterio):
    """Función que filtra una lista de números según un criterio (par o impar)."""
    return list(filter(criterio, lista))

# Ejemplo de uso de la función filtrar_numeros
numeros = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
numeros_pares = filtrar_numeros(numeros, es_par)
numeros_impares = filtrar_numeros(numeros, es_impar)
print(f"Números pares: {numeros_pares}")
print(f"Números impares: {numeros_impares}")



Números pares: [2, 4, 6, 8, 10]
Números impares: [1, 3, 5, 7, 9]


**Buenas prácticas**:
1. **Nomenclatura y documentación.** Nombres descriptivos (p. ej., calcularInteres, validarTransaccion) y comentarios breves que expliquen qué entra, qué sale y qué hace la función. 
2. **Testeo y depuración.** Diseñar funciones testeables en aislamiento (pruebas unitarias) y refactorizar de forma iterativa para eliminar redundancias. ***Validación de Datos***, por ejemplo, comprueba si la lista tiene elementos (cantidad > 0) para evitar dividir entre cero, devolviendo 0 si está vacía.
3. **Reutilización y generalización.** Crear funciones genéricas, promover bibliotecas internas reutilizables y evitar dependencias innecesarias.
***Modularidad y Abstracción***, la función se encarga exclusivamente de una tarea clara, el cálculo del promedio, sin mezclar otras operaciones